Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel $\rightarrow$ Restart) and then **run all cells** (in the menubar, select Cell $\rightarrow$ Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name and collaborators below:

In [ ]:
GROUP_NAME = "Gruppe 9"
COLLABORATORS = "Tobias Arndt,Tim Patzak,Franz Wilhelm Weiss"

---

# Exercise 1: Meditator-Wrapper Architecture for Financial Data

## Due Date: 23.11.2025 23:59 o'clock

In this notebook you will build two wrappers and three mediators to integrate index stock data from <a href="https://www.finanzen.net/">https://www.finanzen.net/</a>. We use this architecture to answer 5 questions about the DAX, TecDax and EURO STOXX 50 stock indexes.

To run this notebook, you only need the pre-installed packages in your `infoint` conda environment. You are allowed to use external libraries that may be installed using ``conda`` or ``pip``.

**The notebook contains 10 tasks for you to solve.**

Wrapper

1. Task: Wrap index components
2. Task: Wrap stock indicators

Mediators

3. Task: Mediate index stock indicators
4. Task: Mediate "cheap" index stock indicators
5. Task: Mediate "overlapping" index stock indicators

Application

6. Task: Retrieve "really large" DAX caps
7. Task: Retrieve "cheapest" EURO STOXX stocks
8. Task: Retrieve "cheap" EURO STOXX "high dividend" stocks
9. Task: Retrieve overlapping DAX / EURO STOXX stock prices
10. Task: Retrieve "cheap" and overlapping DAX / TecDAX stocks


**You should at least pass the implemented tests before handing in your results.**


<div class="alert alert-success alertsuccess" style="margin-top: 20px">
[Tip]: To execute the Python code in the code cell below, click on the cell to select it and press <kbd>Shift</kbd> + <kbd>Enter</kbd>.
</div>


# Financial Data
The internet contains vast amounts of information on financial instruments like stocks. Indexes, like the <a href="https://www.finanzen.net/index/dax">DAX</a>, reflect a collection of stocks based on specific criteria. In this notebook, we want to use the  website <a href="https://www.finanzen.net/">https://www.finanzen.net/</a> to integrate index stock data to be able to answer related queries.

In [ ]:
import os, re
import pandas as pd

## Retrieve HTML from websites

<div class="alert alert-success alertsuccess" style="margin-top: 20px">

The following function retrives the HTML code for a given url from a local cache. You should use this function for the wrappers in Task 1 \& 2. You can then navigate the HTML with librariries like <a href="https://www.crummy.com/software/BeautifulSoup/">BeautifulSoup4</a> to extract information.
    
</div>

*Hint: You can install BeautifulSoup4 with the command: ``conda install -c anaconda beautifulsoup4``*

In [ ]:
html_store = dict()

def retrieve_html(url):
    path = url.replace("https://www.finanzen.net/", "").strip("/")
    file_path = os.path.join("html", f"{path}.html")

    if file_path in html_store:
        return html_store[file_path]

    # cache for reuse
    if os.path.exists(file_path):
        with open(file_path, "rb") as f:
            html = f.read()
            html_store[file_path] = html
            return html

    raise ValueError(f"{url} ({filename}) is not cached locally.")

## Task 1: Wrap index components

<div class="alert alert-success alertsuccess" style="margin-top: 20px">

In this task you should return the stock names and stock urls for a given index in a pandas dataframe. As an input, you get an index URL (e.g. <a href="https://www.finanzen.net/index/dax">https://www.finanzen.net/index/dax</a>). Your function should retrieve the HTML for this url and extract the corresponding stock names and stock urls. As an output it should then return the stock names and urls in a pandas dataframe like:
    
```python
     Stock                                            URL
0   adidas   https://www.finanzen.net/aktien/adidas-aktie
1   Airbus   https://www.finanzen.net/aktien/airbus-aktie
2  Allianz  https://www.finanzen.net/aktien/allianz-aktie
3     BASF     https://www.finanzen.net/aktien/basf-aktie
4    Bayer    https://www.finanzen.net/aktien/bayer-aktie
...
```    

    
Implement the following method

```python
def wrap_index_components(url):

```

that returns a pandas dataframe with stock names and stock urls.

</div>

*Hint: BeautifulSoup4 can find HTML elements with the command: ``find("div", id="some_id")``*

In [ ]:
def wrap_index_components(url):
    from bs4 import BeautifulSoup
    df = list()
    html = retrieve_html(url)#
    soup = BeautifulSoup(html, 'html.parser')
    elements = soup.find('div', id='IndexShareListValues').find_all('a')
    for element in elements:
        URL = "https://www.finanzen.net" + element.get('href')
        Stock = element.get('href').removeprefix("/aktien/").removesuffix("-aktie")
        if URL == "https://www.finanzen.net/aktien/loréal-aktie":
           URL = "https://www.finanzen.net/aktien/loreal-aktie"
        df.append([Stock,URL])
    return pd.DataFrame.from_records(df, index=None, columns=["Stock", "URL"])

In [ ]:
dax = wrap_index_components("https://www.finanzen.net/index/dax")
print(dax.head())

assert type(dax) == pd.DataFrame
assert all(dax.columns == ['Stock', 'URL'])
assert dax.shape[0] == 40

## Task 2: Wrap stock indicators

<div class="alert alert-success alertsuccess" style="margin-top: 20px">

In this task you should return different stock indicators for a given stock as a tupel. As an input, you get a stock URL (e.g. <a href="https://www.finanzen.net/aktien/allianz-aktie">https://www.finanzen.net/aktien/allianz-aktie</a>). Your function should retrieve the HTML for this url and extract the following attributes:
    
1. Stock Name (String)
2. WKN Number (String)
3. Amount of Stocks (Float, in million)
4. Market Capitalisation (Float, in billion)
5. Price Earnings Ratio (KGV) (Float, for the year 2024)
6. Dividend Return in % (Float, for the year 2024)
    
As an output, the function should then return these 6 attributes as a tupel like:
    
```python
    ('Allianz', '840400', 385.92, 135.5, 11.74, 5.2)
```

Implement the following method

```python
def wrap_stock_indicators(url):

```

that returns the 6 stock attributes.

</div>

*Hint: BeautifulSoup4 can find HTML text in tables with the command: ``find("td", string="some text")``. You can traverse the Parsetree using ```find_next('td')```.*

In [ ]:
def wrap_stock_indicators(url):
    from bs4 import BeautifulSoup
    name = url.removeprefix("https://www.finanzen.net/aktien/").removesuffix("-aktie")
    html = retrieve_html(url)
    soup = BeautifulSoup(html, 'html.parser')
    wkn = soup.find('td', class_='table__td font-whitespace-nowrap').find("span").get("data-sg-copy")
    market_cap = float(soup.find("span", class_="badge__value").text.removesuffix(" Mrd. EUR").replace(",", "."))

    price_earnings_ratio = soup.find("article", title="Kurs/Gewinn Verhältnis")
    if price_earnings_ratio == None:
        price_earnings_ratio = float(soup.find("div", class_="grid__item-6 grid__item-12--md").find_all("tr", class_="table__tr")[5].find_all(
            "td", class_="table__td")[1].text.replace(",", ".").replace("-", "0"))
    else:
        price_earnings_ratio = float(price_earnings_ratio.text.removeprefix("KGV").replace(",", "."))
    dividend = float(soup.find("span", title="Dividendenrendite").text.removeprefix("Div. Rendite").removesuffix("%").replace(",", "."))
    stock_amount = float(soup.find("div", class_= "grid__item-6 grid__item-12--md").find_all("tr", class_="table__tr")[1].find_all("td", class_="table__td")[1].text.replace(".", "").replace(",", "."))

    return name, wkn, stock_amount, market_cap, price_earnings_ratio, dividend

In [ ]:
allianz = wrap_stock_indicators("https://www.finanzen.net/aktien/allianz-aktie")
print(allianz)

assert len(allianz) == 6
assert all(type(_) == str for _ in allianz[:2])
assert all(type(_) == float for _ in allianz[2:])

# Milestone 1

Congratulations, at this milestone you should have learned about:

- how to retrieve web page html code
- how to navigate html content
- how to wrap web sources

Going on, we will use the wrappers from Task 1 \& 2 to integrate the data using mediators.

## Task 3: Mediate index stock indicators

<div class="alert alert-success alertsuccess" style="margin-top: 20px">

In this task you should return all stock indicators (Task 2) for a given index (Task 1) as a pandas dataframe. As an input, you get an index URL (e.g. <a href="https://www.finanzen.net/index/dax">https://www.finanzen.net/index/dax</a>). Your function should then use the wrappers from Task 1 \& 2 to get all stock indicators for the given index and return them as a pandas dataframe like:
    
```python
     Stock     WKN    Stocks  Market Cap     PE  Dividend
0   adidas  A1EWWW    178.55       34.80  55.34      0.84
1   Airbus  938914    791.30      161.19  28.89      1.29
2  Allianz  840400    385.92      135.50  11.74      5.20
3     BASF  BASF11    893.85       39.13  29.20      5.30
4    Bayer  BAY001    982.42       26.95   0.00      0.57
...
```

Implement the following method

```python
def mediate_index_stock_indicators(url):

```

that returns the 6 stock attributes for all stocks from the given index.

</div>

*Hint: You can use tqdm (```from tqdm import tqdm```) and wrap lists in it ```tqdm(some_list)``` to see a progress bar how fast the list is iterated.*

In [ ]:
def mediate_index_stock_indicators(url):
    df = list()
    df_components = wrap_index_components(url)
    for item in df_components["URL"]:
        df.append(list(wrap_stock_indicators(item)))
    return pd.DataFrame.from_records(
        df,
        index=None,
        columns=["Stock", "WKN", "# Stocks", "Market Cap", "PE", "Dividend"]
    )

In [ ]:
dax_stock_indicators = mediate_index_stock_indicators("https://www.finanzen.net/index/dax")
print(dax_stock_indicators.head())

assert type(dax_stock_indicators) == pd.DataFrame
assert dax_stock_indicators.shape[0] == 40
assert all(dax_stock_indicators.columns == ['Stock', 'WKN', '# Stocks', 'Market Cap', 'PE', 'Dividend'])

for _, indicators in dax_stock_indicators.iterrows():
    assert len(indicators) == 6
    assert all(type(_) == str for _ in indicators[:2])
    assert all(type(_) == float for _ in indicators[2:])

## Task 4: Mediate "cheap" index stock indicators

<div class="alert alert-success alertsuccess" style="margin-top: 20px">

In this task you should return all "cheap" stock indicators (Task 2) for a given index (Task 1) as a pandas dataframe. We consider a stock as "cheap", if its price earnings ratio (<a href="https://www.finanzen.net/ratgeber/kgv-kurs-gewinn-verhaeltnis-berechnen">KGV</a>) is less than 15. As an input, you get an index URL (e.g. <a href="https://www.finanzen.net/index/dax">https://www.finanzen.net/index/dax</a>). Your function should then use the mediator from Task 3 to get all stock indicators for the given index and return the "cheap" ones as a pandas dataframe like:
    
```python
         Stock     WKN    Stocks  Market Cap     PE  Dividend
2      Allianz  840400    385.92      135.50  11.74      5.20
4        Bayer  BAY001    982.42       26.95   0.00      0.57
6          BMW  519000    559.37       49.48   6.80      5.44
8  Commerzbank  CBK100   1127.50       34.16   7.58      4.13
9  Continental  543900    200.01       12.45  11.10      3.86
...
```

Implement the following method

```python
def mediate_cheap_index_stock_indicators(url, cheap_pe_threshold=15):

```

that returns the 6 stock attributes for all "cheap" stocks from the given index.

</div>

*Hint: You can select conditions on pandas dataframes using ```df.where(some condition)```.*

In [ ]:
def mediate_cheap_index_stock_indicators(url, cheap_pe_threshold=15):
    df = mediate_index_stock_indicators(url)
    df = df[df["PE"] < cheap_pe_threshold]
    return df


In [ ]:
cheap_dax_stock_indicators = mediate_cheap_index_stock_indicators("https://www.finanzen.net/index/dax")
print(cheap_dax_stock_indicators.head())

assert type(cheap_dax_stock_indicators) == pd.DataFrame
assert cheap_dax_stock_indicators.shape[0] <= 40
assert all(cheap_dax_stock_indicators.columns == ['Stock', 'WKN', '# Stocks', 'Market Cap', 'PE', 'Dividend'])

for _, indicators in cheap_dax_stock_indicators.iterrows():
    assert len(indicators) == 6
    assert all(type(_) == str for _ in indicators[:2])
    assert all(type(_) == float for _ in indicators[2:])

## Task 5: Mediate "overlapping" index stock indicators

<div class="alert alert-success alertsuccess" style="margin-top: 20px">

In this task you should return all "overlapping" stock indicators (Task 2) from two given indices (Task 1) as a pandas dataframe. We consider a stock as "overlapping", if it is part of the first as well as the second index. As an input, you get two index URLs (e.g. <a href="https://www.finanzen.net/index/dax">https://www.finanzen.net/index/dax</a> and <a href="https://www.finanzen.net/index/euro_stoxx_50">https://www.finanzen.net/index/euro_stoxx_50</a>). Your function should then use the wrappers from Task 1 \& 2 to get all overlapping stock indicators for the given indices and return them as a pandas dataframe like:
    
```python
     Stock     WKN  # Stocks  Market Cap      PE  Dividend
0   adidas  A1EWWW    178.55       34.80  55.34      0.84
1   Airbus  938914    791.30      161.19  28.89      1.29
2  Allianz  840400    385.92      135.50  11.74      5.20
3     BASF  BASF11    893.85       39.13  29.20      5.30
4    Bayer  BAY001    982.42       26.95   0.00      0.57
...
```

Implement the following method

```python
def mediate_overlap_index_stock_indicators(url1, url2):

```

that returns the 6 stock attributes for all "overlapping" stocks from the given indices.

</div>

*Hint: You can check overlaps in two dataframes with ```df1.ATTR.isin(df2.ATTR)```.*

In [ ]:
def mediate_overlap_index_stock_indicators(url1, url2):
    df1 = wrap_index_components(url1)
    df2 = wrap_index_components(url2)

    tmp_set =set(df1['Stock'])
    tmp_set2 = set(df2['Stock'])
    stocks = list(tmp_set.intersection(tmp_set2))


    df_overlap = []
    df = df1[df1['Stock'].isin(stocks)]
    for item in df["URL"]:
        df_overlap.append(list(wrap_stock_indicators(item)))

    return pd.DataFrame.from_records(
        df_overlap,
        index=None,
        columns=["Stock", "WKN", "# Stocks", "Market Cap", "PE", "Dividend"]
    )


In [ ]:
url1 = "https://www.finanzen.net/index/dax"
url2 = "https://www.finanzen.net/index/euro_stoxx_50"

overlap_stock_indicators = mediate_overlap_index_stock_indicators(url1, url2)
print(overlap_stock_indicators.head())

assert type(overlap_stock_indicators) == pd.DataFrame
assert overlap_stock_indicators.shape[0] <= 40
assert all(overlap_stock_indicators.columns == ['Stock', 'WKN', '# Stocks', 'Market Cap', 'PE', 'Dividend'])

for _, indicators in overlap_stock_indicators.iterrows():
    assert len(indicators) == 6
    assert all(type(_) == str for _ in indicators[:2])
    assert all(type(_) == float for _ in indicators[2:])

# Milestone 2

Congratulations, at this milestone you should have learned about:

- how to use wrappers to retrieve web content
- how to integrate data from multiple sources
- how to restrict integrated data with domain knowledge

Going on, we will use the mediators from Task 3-5 to answer domain-specific queries.

## Task 6: Retrieve "really large" DAX caps

<div class="alert alert-success alertsuccess" style="margin-top: 20px">

In this task you should return all "really large" DAX caps. We consider a stock to be a "really large cap" if its amount of stocks is greater than 1 billion and its market capitilisation is greater than 50 billion. Your function should use the mediators from Task 3-5 to get all "really large" DAX caps and return them as a pandas dataframe like:
    
```python
                   Stock     WKN   Stocks  Market Cap     PE  Dividend
11         Deutsche Bank  514000   1926.15       55.90   8.80      4.09
13      Deutsche Telekom  555750   4819.16      144.62  12.73      3.12
31                   SAP  716460   1164.60      277.87  88.20      0.99
35  Siemens Healthineers  SHL100   1121.56       54.25  31.01      1.76
```

Implement the following method

```python
def retrieve_really_large_dax_caps():

```

that returns the 6 stock attributes for all "really large" DAX caps.

</div>

In [ ]:
def retrieve_really_large_dax_caps():
    url = "https://www.finanzen.net/index/dax"
    df = mediate_index_stock_indicators(url)
    df = df[(df["# Stocks"] > 1000) & (df["Market Cap"]>50) ]
    return df

In [ ]:
really_large_dax_caps = retrieve_really_large_dax_caps()
print(really_large_dax_caps)

assert type(really_large_dax_caps) == pd.DataFrame
assert really_large_dax_caps.shape[0] < 10
assert all(really_large_dax_caps.columns == ['Stock', 'WKN', '# Stocks', 'Market Cap', 'PE', 'Dividend'])

## Task 7: Retrieve "cheapest" EURO STOXX stocks

<div class="alert alert-success alertsuccess" style="margin-top: 20px">

In this task you should return the 10 cheapest EURO STOXX 50 stocks in ascending order. We assess whether a stock is cheap by its price earnings ratio. Your function should use the mediators from Task 3-5 to get "cheap" EURO STOXX 50 stocks, determine the 10 cheapest ones and return them in ascending order (by PE ratio) as a pandas dataframe like:
    
```python
                               Stock     WKN    Stocks  Market Cap     PE  Dividend
11                             Bayer  BAY001    982.42       26.95  0.00   0.57
48               Volkswagen (VW) vz.  766403    206.20       45.61  4.16   7.14
31  Mercedes-Benz Group (ex Daimler)  710000    962.90       51.83  5.28   7.99
12                              BBVA  875773   5753.29       99.50  5.64   6.00
39                         Santander  858872  14869.77      125.50  5.79   3.81
...
```

Implement the following method

```python
def retrieve_cheapest_eurostoxx_stocks():

```

that returns the 6 stock attributes for the 10 "cheapest" EURO STOXX 50 stocks in ascending order.

</div>

*Hint: You can sort dataframes using ```df.sort_values(by="attribute_name")```.*

In [ ]:
def retrieve_cheapest_eurostoxx_stocks():
    url = "https://www.finanzen.net/index/euro_stoxx_50"
    df = mediate_index_stock_indicators(url)
    df = df.sort_values(by="PE").head(10)
    return df


In [ ]:
cheapest_eurostoxx_stocks = retrieve_cheapest_eurostoxx_stocks()
print(cheapest_eurostoxx_stocks)

assert type(cheapest_eurostoxx_stocks) == pd.DataFrame
assert cheapest_eurostoxx_stocks.shape[0] == 10
assert all(cheapest_eurostoxx_stocks.columns == ['Stock', 'WKN', '# Stocks', 'Market Cap', 'PE', 'Dividend'])

## Task 8: Retrieve "cheap" EURO STOXX "high dividend" stocks

<div class="alert alert-success alertsuccess" style="margin-top: 20px">

In this task you should return cheap EURO STOXX 50 stocks that return more than 3\% dividend and sort them in descending order by dividend. A stock is cheap if its the price earnings ratio is less than 15 (compare Task 4). Your function should use the mediators from Task 3-5 to get the  "cheap" EURO STOXX 50 stocks, select the ones with a dividend larger than 3\%, sort them in descending order and return them as a pandas dataframe like:
    
```python
                                       Stock     WKN    Stocks  Market Cap     PE        Dividend
33                Nordea Bank Abp Registered  A2N6F4   3439.18       50.14     7.30      8.95
28                           Intesa Sanpaolo  850605  17778.07       97.10     8.03      8.83
14                               BNP Paribas  887771   1103.81       76.27     6.19      8.09
31          Mercedes-Benz Group (ex Daimler)  710000    962.90       51.83     5.28      7.99
48                       Volkswagen (VW) vz.  766403    206.20       45.61     4.16      7.14
...
```

Implement the following method

```python
def retrieve_cheap_eurostoxx_high_divident_stocks():

```

that returns the 6 stock attributes for cheap EURO STOXX 50 stocks that return more than 3\% dividend in descending order.

</div>

*Hint: You can drop NaN rows in dataframes using ```df.dropna()```.*

In [ ]:
def retrieve_cheap_eurostoxx_high_divident_stocks():
    url = "https://www.finanzen.net/index/euro_stoxx_50"
    df = mediate_index_stock_indicators(url)
    df = df[(df["PE"] < 15) & (df["Dividend"]>3) ]
    df = df.sort_values(ascending=False,by="Dividend")
    return df

In [ ]:
cheap_eurostoxx_high_divident_stocks = retrieve_cheap_eurostoxx_high_divident_stocks()
print(cheap_eurostoxx_high_divident_stocks)

assert type(cheap_eurostoxx_high_divident_stocks) == pd.DataFrame
assert cheap_eurostoxx_high_divident_stocks.shape[0] < 20
assert all(cheap_eurostoxx_high_divident_stocks.columns == ['Stock', 'WKN', '# Stocks', 'Market Cap', 'PE', 'Dividend'])

## Task 9: Retrieve overlapping DAX / EURO STOXX stock prices

<div class="alert alert-success alertsuccess" style="margin-top: 20px">

In this task you should return 10 cheapest stock names and prices that are listed in both the DAX as well as the EURO STOXX 50 sorted by price in ascending order. In this task, we consider a stock "cheap" or "expensive" based on its stock price. You can determine a stock's price by dividing its market capitalisation by the amount of issued stocks. Your function should use the mediators from Task 3-5 to get the "overlapping" DAX and EURO STOXX 50 stocks, calculate their price, determine the 10 cheapest stocks and return them in ascending order as a pandas dataframe like:
    
```python
                           Stock      Price
4                          Bayer  27.432259
6                  Deutsche Bank  29.021623
8               Deutsche Telekom  30.009379
10                      Infineon  34.098956
9   DHL Group (ex Deutsche Post)  39.908135
...
```

Implement the following method

```python
def retrieve_dax_eurostoxx_stock_prices():

```

that returns the stock name and price for the 10 cheapest overlapping DAX and EURO STOXX 50 stocks in ascending order.

</div>

*Hint: You can create empty dataframes using ```pd.DataFrame()``` and set columns from other dataframes using ```df2["Attribute"] = df1["Attribute"]```.*

In [ ]:
def retrieve_dax_eurostoxx_stock_prices():
    url1 = "https://www.finanzen.net/index/dax"
    url2 = "https://www.finanzen.net/index/euro_stoxx_50"


    df = mediate_overlap_index_stock_indicators(url1, url2)
    df["Price"] = df["Market Cap"]* 1000 / df["# Stocks"]
    df =  df.sort_values(by="Price")
    return df[['Stock', 'Price']].head(10)

In [ ]:
dax_eurostoxx_stock_prices = retrieve_dax_eurostoxx_stock_prices()
print(dax_eurostoxx_stock_prices.head())

assert type(dax_eurostoxx_stock_prices) == pd.DataFrame
assert dax_eurostoxx_stock_prices.shape[0] == 10
assert all(dax_eurostoxx_stock_prices.columns == ['Stock', 'Price'])

## Task 10: Retrieve "cheap" and overlapping DAX / TecDAX stocks

<div class="alert alert-success alertsuccess" style="margin-top: 20px">

In this task you should return cheap stocks that are listed in both the DAX as well as the TecDAX. A stock is cheap if its the price earnings ratio is less than 15 (compare Task 4). Your function should use the mediators from Task 3-5 to get the "overlapping" DAX and TecDAX stocks, determine the "cheap" ones and return them as a pandas dataframe like:
    
```python
              Stock     WKN  # Stocks  Market Cap     PE  Dividend
0  Deutsche Telekom  555750   4819.16      144.62  12.73      3.12
```

Implement the following method

```python
def retrieve_cheap_dax_tdax_stocks():

```

that returns cheap DAX and TecDAX stocks.

</div>

In [ ]:
def retrieve_cheap_dax_tdax_stocks(cheap_pe_threshold=15):
    url1 = "https://www.finanzen.net/index/dax"
    url2 = "https://www.finanzen.net/index/tecdax"
    df = mediate_overlap_index_stock_indicators(url1, url2)
    df = df[df["PE"] < cheap_pe_threshold]
    return df


In [ ]:
cheap_dax_tdax_stocks = retrieve_cheap_dax_tdax_stocks()
print(cheap_dax_tdax_stocks)

assert type(cheap_dax_tdax_stocks) == pd.DataFrame
assert cheap_dax_tdax_stocks.shape[0] < 5
assert all(cheap_dax_tdax_stocks.columns == ['Stock', 'WKN', '# Stocks', 'Market Cap', 'PE', 'Dividend'])

# Milestone 3

Congratulations, at this milestone you should have learned about:

- how to use mediators to answer queries
- how to select, project and sort the result data
- how to use dataframes

Please upload this notebook (the .ipynb file) in Moodle until **23.11.2025 23:59 o'clock.**